# CP1 Week 8 -- Strings: Parsing Data

**Course:** Computer Programming 1 (CP1) | **Session:** 5 hours

## Learning Objectives

1. Use string methods: `split()`, `strip()`, `join()`, `replace()`
2. Parse structured text into data records
3. Handle multiple data formats (CSV, key:value, key=value)
4. Build a robust parser for your pipeline

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: String Methods

Strings are the format data arrives in from files. Before you can do math
or analysis, you must **parse** (break apart) the strings into numbers.

In [ ]:
text = "  Hello, World!  "

print("Original:", repr(text))
print("strip():", repr(text.strip()))
print("lower():", text.strip().lower())
print("upper():", text.strip().upper())
print("split(','):", text.strip().split(","))
print("replace('World', 'Python'):", text.strip().replace("World", "Python"))
print("startswith('  He'):", text.startswith("  He"))
print("endswith('!  '):", text.endswith("!  "))
print("'World' in text:", "World" in text)

**Expected Output:**
```
Original: '  Hello, World!  '
strip(): 'Hello, World!'
lower(): 'hello, world!'
upper(): 'HELLO, WORLD!'
split(','): ['Hello', ' World!']
replace('World', 'Python'): 'Hello, Python!'
startswith('  He'): True
endswith('!  '): True
'World' in text: True
```

### Part 2: Parsing CSV lines

In [ ]:
def parse_csv_line(line):
    """Parse a comma-separated line into cleaned values."""
    parts = line.strip().split(",")
    return [p.strip() for p in parts]

lines = [
    "sensor_01, 25.3, normal",
    "sensor_02, 88.1, warning",
    "sensor_03, -5, error",
]

print("Parsed records:")
for line in lines:
    parts = parse_csv_line(line)
    print(f"  Name: {parts[0]}, Value: {parts[1]}, Status: {parts[2]}")

**Expected Output:**
```
Parsed records:
  Name: sensor_01, Value: 25.3, Status: normal
  Name: sensor_02, Value: 88.1, Status: warning
  Name: sensor_03, Value: -5, Status: error
```

### Part 3: Multi-format parser

In [ ]:
def parse_reading(text):
    """Parse a sensor reading from various formats.

    Supported: "name: value", "name=value", "name,value"
    """
    text = text.strip()

    if ": " in text:
        parts = text.split(": ", 1)
    elif "=" in text:
        parts = text.split("=", 1)
    elif "," in text:
        parts = text.split(",", 1)
    else:
        return None

    if len(parts) != 2:
        return None

    name = parts[0].strip()
    try:
        value = float(parts[1].strip())
    except ValueError:
        return None

    return {"name": name, "value": value}

# Test
inputs = ["temp: 25.3", "rpm=1500", "vib,12.5", "bad data", "too:many:colons"]
for text in inputs:
    result = parse_reading(text)
    print(f"  '{text}' -> {result}")

**Expected Output:**
```
  'temp: 25.3' -> {'name': 'temp', 'value': 25.3}
  'rpm=1500' -> {'name': 'rpm', 'value': 1500.0}
  'vib,12.5' -> {'name': 'vib', 'value': 12.5}
  'bad data' -> None
  'too:many:colons' -> {'name': 'too', 'value': None}
```

### Try It Yourself

In [ ]:
# TODO: Parse these log entries into a list of dictionaries.
# Each log has format: "YYYY-MM-DD HH:MM:SS | LEVEL | message"

logs = [
    "2024-01-15 10:30:00 | INFO | System started",
    "2024-01-15 10:31:05 | WARNING | Temperature high",
    "2024-01-15 10:32:10 | ERROR | Sensor disconnected",
]

parsed = []
for log in logs:
    # Split on " | " to get 3 parts
    parts = log.split(" | ")
    entry = {
        "timestamp": parts[0],
        "level": parts[1],
        "message": parts[2],
    }
    parsed.append(entry)

for entry in parsed:
    print(f"  [{entry['level']:>7}] {entry['timestamp']} - {entry['message']}")

**Expected Output:**
```
  [   INFO] 2024-01-15 10:30:00 - System started
  [WARNING] 2024-01-15 10:31:05 - Temperature high
  [  ERROR] 2024-01-15 10:32:10 - Sensor disconnected
```

### Common Mistakes with Strings

| Mistake | What happens | Fix |
|---------|-------------|-----|
| split() with wrong delimiter | Gets one big string | Check what separator your data uses |
| Forgetting strip() | Leading/trailing spaces | Always `strip()` after `split()` |
| Not handling empty strings | Crashes on empty | Check `if text.strip():` before parsing |
| Wrong split count | `"a:b:c".split(":")` gives 3 parts | Use `split(":", 1)` to limit splits |

### Debugging Tip

When your parser gives wrong results, use `repr()` to see hidden characters:
```python
text = "  hello  "
print(repr(text))   # '  hello  '  -- shows the spaces!
```

### Part 4: join() -- putting strings back together

In [ ]:
words = ["Hello", "World", "from", "Python"]
print(", ".join(words))
print(" | ".join(words))
print("\n".join(words))

# Common use: creating CSV lines
headers = ["id", "name", "value", "status"]
row = ["1", "sensor_01", "25.3", "ok"]
csv_line = ",".join(row)
print(f"CSV: {csv_line}")

**Expected Output:**
```
Hello, World, from, Python
Hello | World | from | Python
Hello
World
from
Python
CSV: 1,sensor_01,25.3,ok
```

### Example -- Building a complete CSV parser

In [ ]:
def parse_csv_string(csv_text):
    """Parse a multi-line CSV string into a list of dictionaries."""
    lines = csv_text.strip().split("\n")
    if len(lines) < 2:
        return []

    headers = [h.strip() for h in lines[0].split(",")]
    rows = []

    for line_num, line in enumerate(lines[1:], start=2):
        parts = [p.strip() for p in line.split(",")]
        if len(parts) != len(headers):
            print(f"  Warning: line {line_num} has {len(parts)} fields, expected {len(headers)}")
            continue
        row = {}
        for header, value in zip(headers, parts):
            row[header] = value
        rows.append(row)

    return rows

csv_data = """name, value, unit
temp_01, 25.3, celsius
temp_02, 88.1, celsius
pressure, 101.3, kPa
humidity, 45, percent"""

parsed = parse_csv_string(csv_data)
print(f"Parsed {len(parsed)} rows:")
for row in parsed:
    print(f"  {row}")

**Expected Output:**
```
Parsed 4 rows:
  {'name': 'temp_01', 'value': '25.3', 'unit': 'celsius'}
  {'name': 'temp_02', 'value': '88.1', 'unit': 'celsius'}
  {'name': 'pressure', 'value': '101.3', 'unit': 'kPa'}
  {'name': 'humidity', 'value': '45', 'unit': 'percent'}
```

### Why This Matters for Your Pipeline

String parsing is what `load_data()` does when reading from files.
Raw data arrives as text -- every CSV cell is a string. Your parser
must:
1. Split the text into rows and columns
2. Strip whitespace from each field
3. Handle missing fields and malformed lines
4. Convert numeric strings to actual numbers

This is where Weeks 3 (conditionals), 5 (loops), and 8 (strings) all
come together!

---
## Key Takeaways -- Week 8

1. **`strip()`** removes whitespace; **`split()`** breaks strings apart
2. **`join()`** combines a list into a string
3. **`repr()`** shows hidden characters for debugging
4. Real data comes in many formats -- build parsers that handle them
5. Always handle parsing failures gracefully (return None, skip, log)

---
## Homework

### Review (R1-R4)

In [ ]:
# R1: What does "hello, world".split(",") return?
# R2: What does " hi ".strip() return?
# R3: What does ", ".join(["a","b","c"]) return?
# R4: Why is parsing important for your pipeline?

### Practice (P1-P5)

In [ ]:
# P1: Split a sentence into words and count them.
sentence = "The quick brown fox jumps over the lazy dog"


In [ ]:
# P2: Parse these log lines into dicts with keys: timestamp, level, message
logs = [
    "2024-01-15 10:30:00 | INFO | System started",
    "2024-01-15 10:31:05 | WARNING | Temperature high",
    "2024-01-15 10:32:10 | ERROR | Sensor disconnected",
]


In [ ]:
# P3: Parse a multi-line CSV string into a list of dicts.
csv_text = """name,value,unit
temp_01,25.3,celsius
temp_02,88.1,celsius
pressure_01,101.3,kPa"""


In [ ]:
# P4: Write a function that validates email addresses.
# Must contain @, text before and after @, dot after @.


In [ ]:
# P5: Write a function that auto-detects the delimiter
# (comma, tab, pipe, semicolon) in a data line.


### Challenge (C1-C3)

In [ ]:
# C1: Parse a key=value config file format into a dictionary.
config_text = """project_name=my_sensor
track=robotics
threshold=60.0
version=v1"""


In [ ]:
# C2: Write a CSV parser that handles quoted fields:
# 'name,"city, state",value' -> ["name", "city, state", "value"]


In [ ]:
# C3: Write a function that reformats data from one delimiter to another.


### Mini-Project

In [ ]:
# M1: Multi-Format Data Loader
# Write a load_data() that can read:
# 1. CSV format: "name,value,status"
# 2. Key-value: "name: value"
# 3. Tab-separated: "name\tvalue\tstatus"
# Auto-detect the format and parse accordingly.
# Return a list of dictionaries.

test_data = [
    # CSV
    "sensor_01,25.3,ok",
    "sensor_02,88.1,warning",
    # Key-value
    "temp: 30.5",
    "rpm: 1500",
]


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)